# The Dzud — Teaching a Machine to Predict Disaster

<a target="_blank" href="https://jupyter-k12.org?github=simonguest/codercub/blob/main/labs/03/notebooks/regression.ipynb">
  <img src="https://img.shields.io/badge/Open_in-Jupyter_K--12-blue" alt="Open In Jupyter K-12"/>
</a>

In the previous two notebooks you found the disasters and identified their causes. You know that winter temperature is the strongest predictor of livestock mortality, and that summer drought adds independent explanatory power on top of it.

Now comes the question that makes this practically useful: can a machine learn that relationship from historical data and use it to make predictions about years it has never seen?

That is linear regression — and it is also, at its core, the same idea behind every model you will work with this week, including the micro:bit gesture detector on Thursday.

By the end of this notebook you will have trained two models, compared their accuracy, and used the better one to predict what would happen to a Mongolian herding family given a specific forecast of winter temperature and summer drought.

## Step 1: Load the data and imports

We add one new import today: `scikit-learn`, the most widely used machine learning library in Python. We will use just two things from it — a `LinearRegression` model and a function to split data into training and test sets.

In [ ]:
import httpx
import io
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

url = "https://raw.githubusercontent.com/simonguest/codercub/main/labs/03/mongolia_dzud_1990_2013.csv"
response = httpx.get(url)
df = pd.read_csv(io.StringIO(response.text))

print(f"Loaded {len(df)} rows")
df.head()

## Step 2: The idea behind linear regression

Before writing any model code, it is worth understanding what linear regression actually does.

In the previous notebook you found that colder winters tend to produce higher mortality. If you drew a straight line through that scatter plot that best fits the data, you would have a linear regression model. Given a new winter temperature value, the line tells you the predicted mortality.

The line has two properties:
- A **slope** — how much mortality changes for each one-degree change in temperature
- An **intercept** — where the line crosses the y-axis (the predicted mortality at 0°C)

A model with two input variables (temperature and drought) fits a flat plane through the data instead of a line — the same idea, one dimension higher.

Scikit-learn finds the slope and intercept automatically by minimising the total distance between the line (or plane) and all the actual data points. You do not need to calculate this by hand.

What do you think will happen to the predicted mortality if winter temperature goes down by 5°C? What about if summer drought index goes up from 0.2 to 0.8?

Before training the model: if winter temperature drops by 5°C, what do you predict happens to mortality? What if the summer drought index rises from 0.2 to 0.8? Write your prediction — you can check it against the model's coefficients later.

## Step 3: Prepare the data

Before training, we need to decide on three things:
1. Which columns are our **inputs** (features) — the variables the model will learn from
2. Which column is our **output** (target) — the value the model will try to predict
3. How to split the data into a **training set** (data the model learns from) and a **test set** (data we hold back to evaluate it)

Keeping the test set hidden during training is critical. If we evaluated the model on data it had already seen, we would have no idea how well it generalises to genuinely new situations.

In [ ]:
# Input features and target
X = df[["winter_temp_c"]]          # start with one variable only
y = df["mortality_pct"]

# Split: 80% for training, 20% held back for testing
# random_state=42 ensures everyone gets the same split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")
print(f"\nThe model will learn from {len(X_train)} aimag-years")
print(f"and be evaluated on {len(X_test)} aimag-years it has never seen")

## Step 4: Train the first model — temperature only

Three lines of code to train a model. That is all scikit-learn requires.

The `fit` call is where the learning happens — it finds the slope and intercept that best describes the relationship between winter temperature and mortality in the training data.

In [ ]:
model_1 = LinearRegression()
model_1.fit(X_train, y_train)

print(f"Model learned:")
print(f"  Slope (per 1°C change in winter temp): {model_1.coef_[0]:.3f}")
print(f"  Intercept:                             {model_1.intercept_:.3f}")
print()
print(f"Interpretation: for every 1°C colder the winter is,")
print(f"the model predicts mortality increases by {abs(model_1.coef_[0]):.2f} percentage points")

The slope tells you something directly interpretable: each additional degree of winter cold is associated with a specific increase in predicted mortality.

Does that number feel intuitively reasonable given what you saw in the scatter plots?

The model learned a specific slope — how much mortality increases per degree of cold. Does that number feel intuitively reasonable based on the scatter plots you saw earlier? Is it larger or smaller than you expected?

## Step 5: Evaluate the first model

Training a model is straightforward. The harder question is: how good is it?

We evaluate on the **test set** — the 20% of data the model has never seen.

Two metrics:
- **R² (R-squared)**: how much of the variation in mortality does the model explain? Ranges from 0 (explains nothing) to 1 (explains everything perfectly). An R² of 0.7 means the model accounts for 70% of the variation.
- **MAE (Mean Absolute Error)**: on average, how many percentage points is the model's prediction off from the real value?

In [ ]:
y_pred_1 = model_1.predict(X_test)

r2_1  = r2_score(y_test, y_pred_1)
mae_1 = mean_absolute_error(y_test, y_pred_1)

print(f"Model 1 — temperature only")
print(f"  R²:  {r2_1:.3f}  (explains {r2_1*100:.1f}% of the variation in mortality)")
print(f"  MAE: {mae_1:.2f} percentage points average error")

An R² in the range of 0.65–0.75 is a reasonable result for a single-variable model on noisy real-world data. The MAE tells you something more concrete — on average, the model's prediction is off by that many percentage points.

For context, in a year like 2001 when average national mortality was around 23%, an MAE of several percentage points represents a meaningful margin of error for a herder trying to decide whether to move their animals or seek insurance support.

## Step 6: Visualise the first model's predictions

A good way to inspect a model is to plot its predictions against the actual values. If the model were perfect, every point would lie on the diagonal. Points above the diagonal are cases where the model under-predicted mortality; points below are over-predictions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: predicted vs actual
ax = axes[0]
ax.scatter(y_test, y_pred_1, alpha=0.45, s=18, color="steelblue")
lims = [0, max(y_test.max(), y_pred_1.max()) + 2]
ax.plot(lims, lims, "k--", linewidth=1, label="Perfect prediction")
ax.set_xlabel("Actual mortality (%)", fontsize=11)
ax.set_ylabel("Predicted mortality (%)", fontsize=11)
ax.set_title(f"Model 1: temperature only\nR² = {r2_1:.3f}", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

# Right: the regression line over the training data
ax = axes[1]
temp_range = np.linspace(df["winter_temp_c"].min(), df["winter_temp_c"].max(), 200).reshape(-1, 1)
pred_line  = model_1.predict(temp_range)

ax.scatter(df["winter_temp_c"], df["mortality_pct"],
           alpha=0.2, s=12, color="steelblue", label="Observed data")
ax.plot(temp_range, pred_line, color="firebrick", linewidth=2, label="Regression line")
ax.set_xlabel("Winter temperature (°C)", fontsize=11)
ax.set_ylabel("Mortality (%)", fontsize=11)
ax.set_title("The fitted regression line", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

Look at the predicted vs. actual chart on the left. The points should follow the diagonal broadly, but with noticeable scatter — especially at higher mortality values.

The wide spread at the top right is the model struggling with the catastrophic dzud years. What might explain why the temperature-only model has more trouble predicting the worst outcomes?

Look at the predicted vs. actual chart. Where does the model struggle most — at low mortality values, high values, or both? Why do you think the temperature-only model has more trouble predicting the worst dzud years?

## Step 7: Train the second model — temperature and drought

Now we add the summer drought index as a second input feature. The change to the code is minimal — we just add the second column to X. Scikit-learn handles the rest.

In [ ]:
# Two input features this time
X2 = df[["winter_temp_c", "summer_drought_idx"]]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42   # same random_state = same split
)

model_2 = LinearRegression()
model_2.fit(X2_train, y2_train)

print(f"Model 2 learned:")
print(f"  Coefficient for winter_temp_c:        {model_2.coef_[0]:.3f}")
print(f"  Coefficient for summer_drought_idx:   {model_2.coef_[1]:.3f}")
print(f"  Intercept:                            {model_2.intercept_:.3f}")
print()
print(f"Interpretation:")
print(f"  Each 1°C colder  → mortality increases by {abs(model_2.coef_[0]):.2f} pp")
print(f"  Each 0.1 increase in drought index → mortality increases by {model_2.coef_[1]*0.1:.2f} pp")

## Step 8: Evaluate the second model

Let's measure whether adding drought genuinely improves the model.

In [ ]:
y_pred_2 = model_2.predict(X2_test)

r2_2  = r2_score(y2_test, y_pred_2)
mae_2 = mean_absolute_error(y2_test, y_pred_2)

print(f"Model 1 — temperature only")
print(f"  R²:  {r2_1:.3f}   MAE: {mae_1:.2f} pp")
print()
print(f"Model 2 — temperature + drought")
print(f"  R²:  {r2_2:.3f}   MAE: {mae_2:.2f} pp")
print()
r2_improvement  = (r2_2 - r2_1) * 100
mae_improvement = mae_1 - mae_2
print(f"Adding drought index:")
print(f"  R² improved by  {r2_improvement:.1f} percentage points")
print(f"  MAE improved by {mae_improvement:.2f} percentage points")

The R² should improve meaningfully when drought is added — the model now explains more of the variation in mortality. The MAE should decrease, meaning predictions are closer to the actual values on average.

Is the improvement larger or smaller than you expected? Does it change your view of how important the summer drought variable is relative to winter temperature?

Was the improvement from adding drought larger or smaller than you expected? Does it change your view of how important the summer drought variable is compared to winter temperature?

## Step 9: Side-by-side comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_1, y_pred_2],
    [f"Model 1: temperature only\nR² = {r2_1:.3f}, MAE = {mae_1:.2f} pp",
     f"Model 2: temperature + drought\nR² = {r2_2:.3f}, MAE = {mae_2:.2f} pp"]
):
    ax.scatter(y_test, y_pred, alpha=0.45, s=18, color="steelblue")
    lims = [0, max(y_test.max(), max(y_pred)) + 2]
    ax.plot(lims, lims, "k--", linewidth=1, label="Perfect prediction")
    ax.set_xlabel("Actual mortality (%)", fontsize=11)
    ax.set_ylabel("Predicted mortality (%)", fontsize=11)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=9)

plt.suptitle("Predicted vs. actual mortality — model comparison", 
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

The right chart should show points sitting closer to the diagonal — particularly at the high end, where the temperature-only model struggled most.

The improvement at the extremes is especially important: those are the catastrophic dzud years, where accurate prediction has the most practical value for herders and government planners.

## Step 10: Make a prediction

The model is now a tool. Given a forecast of winter temperature and summer drought, it can output a predicted mortality percentage — before the winter even arrives.

Let's use it to answer a concrete question: what does the model predict for three different scenarios a Mongolian herder might face?

In [ ]:
scenarios = pd.DataFrame({
    "scenario":          ["Mild winter, no drought",
                          "Cold winter, no drought",
                          "Cold winter, severe drought"],
    "winter_temp_c":     [-16.0, -24.0, -24.0],
    "summer_drought_idx": [ 0.10,  0.15,  0.75]
})

scenarios["predicted_mortality_pct"] = model_2.predict(
    scenarios[["winter_temp_c", "summer_drought_idx"]]
).round(1)

print(scenarios[["scenario", "winter_temp_c", "summer_drought_idx",
                 "predicted_mortality_pct"]].to_string(index=False))

Look at the difference between the second and third scenarios — the same cold winter, but one is preceded by drought and one is not.

Now try building your own scenario. What inputs would you give the model to produce the highest possible predicted mortality? What about the lowest? Are the model's predictions plausible given what you have learned about dzud?

In [ ]:
# Build your own scenario — change these values and re-run
my_winter_temp  = -20.0    # typical Mongolian winter is around -18 to -22
my_drought      =  0.40    # 0 = no drought, 1 = severe drought

my_scenario = pd.DataFrame({
    "winter_temp_c":     [my_winter_temp],
    "summer_drought_idx": [my_drought]
})

prediction = model_2.predict(my_scenario)[0]
print(f"Winter temperature:  {my_winter_temp}°C")
print(f"Summer drought index: {my_drought}")
print(f"Predicted mortality:  {prediction:.1f}%")
print()

# Put that in context with a hypothetical herd
herd_size = 2_000
predicted_losses = int(herd_size * prediction / 100)
print(f"For a herder with {herd_size:,} animals,")
print(f"that is approximately {predicted_losses:,} animals lost.")

## Step 11: The same idea — a different kind of data

You have just built a model that takes numerical inputs and predicts a numerical output. The process was:

1. Collect labeled examples (historical aimag-years with known outcomes)
2. Split into training and test sets
3. Fit a model to the training data
4. Evaluate on the test set
5. Use the model to predict on new, unseen inputs

On Thursday you will do exactly the same thing with your micro:bit — but instead of aimag climate data, the inputs will be accelerometer readings from your wrist, and instead of predicting mortality, the model will predict which gesture you are making.

The tools will be different (CreateAI rather than scikit-learn), the data will be different (motion rather than climate), and the output will be different (a category rather than a number). But the underlying loop — label examples, train, evaluate, predict — is identical.

That loop is the core of supervised machine learning. You have now run through it completely.

## What you have done across these three notebooks

Across the Day 3 sequence you have worked through a complete data science and machine learning workflow on a real-world problem:

- Loaded and explored a multi-dimensional dataset (Notebook 1)
- Identified the variables that explain an outcome through correlation and visualisation (Notebook 2)
- Trained two linear regression models, compared their accuracy with R² and MAE, and used the better model to make predictions on new scenarios (Notebook 3)

The problem — predicting livestock disaster in Mongolia — is one that researchers and international insurance programmes have worked on for years using exactly these methods. You have replicated the core of that analysis in three notebooks.

Tomorrow you will take the same supervised learning concept and apply it to physical movement data. The jump from climate data to accelerometer data is smaller than it looks.

## Check Your Understanding